In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#  Create flag Parameter

In [0]:
dbutils.widgets.text("Incremental_flag","0")

In [0]:
incremental_flag=dbutils.widgets.get("Incremental_flag")
print(incremental_flag)

1


# Creatiing Dimension Model

In [0]:
df_src=spark.sql(''' SELECT Distinct(Date_ID)
FROM parquet.`abfss://silver@carprojectazurestorage.dfs.core.windows.net/carsales`''')


In [0]:
df_src.display()

Date_ID
DT00029
DT00030
DT00039
DT00078
DT00116
DT00120
DT00124
DT00136
DT00140
DT00164


## dim_sink - Initial and Incremental

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    df_sink=spark.sql('''select dim_date_key,Date_ID
                  from cars_catalog.gold.dim_date
                  ''')
    print("incremental")
else:
    df_sink=spark.sql('''select 1 as dim_date_key,Date_ID
                  from parquet.`abfss://silver@carprojectazurestorage.dfs.core.windows.net/carsales`
                  where 1=0''')
            
df_sink.display()

incremental


dim_date_key,Date_ID
1,DT00029
2,DT00030
3,DT00039
4,DT00078
5,DT00116
6,DT00120
7,DT00124
8,DT00136
9,DT00140
10,DT00164


## filter records

In [0]:
df_filter=df_src.join(df_sink,df_src.Date_ID==df_sink.Date_ID,'left').select(df_src.Date_ID,df_sink.dim_date_key)
df_filter.display()

Date_ID,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10


## Filtering new and old records

### old records

In [0]:
df_filter_old=df_filter.filter(col('dim_date_key').isNotNull())
df_filter_old.display()

Date_ID,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10


### new records

In [0]:
df_filter_new=df_filter.filter(col('dim_date_key').isNull()).select(df_src['Date_ID'])
df_filter_new.display()

Date_ID


# Creating Surrogate Key

**fetch the max surrogate key from existing table**

In [0]:
if incremental_flag=='0':
    max_value=1
else:
    max_value=spark.sql('''select max(dim_date_key) from cars_catalog.gold.dim_date''')
    max_value=max_value.collect()[0][0]
print(max_value)

1156


**create the surrogate key and add it**

In [0]:
df_filter_new=df_filter_new.withColumn('dim_date_key',monotonically_increasing_id()+max_value)
df_filter_new.display()

Date_ID,dim_date_key


### Creating final Df - df_filter_old + df_filter_new

In [0]:
df_final=df_filter_old.union(df_filter_new)
df_final.display()

Date_ID,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10


# SCD Type - I

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    delta_table=DeltaTable.forPath(spark,"abfss://gold@carprojectazurestorage.dfs.core.windows.net/dim_date")
    delta_table .alias('dlt').merge(df_final.alias('s'),'dlt.Date_ID=s.Date_ID')\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_final.write.format('Delta')\
            .mode("overwrite")\
            .option('path','abfss://gold@carprojectazurestorage.dfs.core.windows.net/dim_date')\
            .saveAsTable('cars_catalog.gold.dim_date')


In [0]:
%sql
select * from cars_catalog.gold.dim_date

Date_ID,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10
